In [1]:
from google.adk.agents import Agent


In [2]:
from google.adk.models.lite_llm import LiteLlm

In [3]:
%pip install -q -U google-adk requests

In [4]:
import requests
import logging
import re
from google.adk.agents.callback_context import CallbackContext
from google.adk.models.llm_request import LlmRequest
from google.adk.models.llm_response import LlmResponse
from google.genai import types
from google.adk.models.base_llm import BaseLlm
from google.adk.agents import Agent

6. Add VertexAI


In [5]:
import os
import google.auth
import vertexai

# Retrieve credentials and project information
credentials, project = google.auth.default()

# Configure environment variables for Vertex AI
os.environ["GOOGLE_CLOUD_PROJECT"] = "qwiklabs-gcp-02-29d1792d4bd6"
os.environ["VERTEX_AI_PROJECT"] = "qwiklabs-gcp-02-29d1792d4bd6"
os.environ["VERTEX_AI_LOCATION"] = "us-central1"

# Initialize the Vertex AI SDK
vertexai.init(
    project="qwiklabs-gcp-02-29d1792d4bd6",
    location="us-central1",
    credentials=credentials
)

print("Vertex AI initialized successfully!")

Vertex AI initialized successfully!


In [6]:
from google.adk.models.lite_llm import LiteLlm

# Instantiate the Vertex AI model using the Google ADK wrapper
# This will leverage the Vertex AI project and credentials we configured earlier
MODEL = LiteLlm(model="gemini-2.5-flash")

print("Model 'MODEL' initialized successfully with gemini-2.5-flash!")

Model 'MODEL' initialized successfully with gemini-2.5-flash!


**Add Greeter Agent**

In [7]:

from google.adk.agents import Agent

greeter_agent = Agent(
    name="greeter_agent",
    model=MODEL,
    description="Greets users and introduces the agent system.",
    instruction="""
    You are a friendly AI assistant greeter.

    Welcome the user warmly.

    Introduce the system's capabilities:
    - Answering questions
    - Researching information
    - Reviewing answers for accuracy
    - Improving responses

    Keep greetings short and professional.

    Do not conduct research.
    """
)

print("Greeter Agent created!")


Greeter Agent created!


7. Add Google Search

In [8]:
from google.adk.tools import google_search

print("Google Search tool imported successfully!")

Google Search tool imported successfully!


8. Create Google Search Agent

In [9]:


search_agent = Agent(
    name="search_agent",
    model=MODEL,
    description="Researches questions using Google Search.",
    instruction="""
    You are the Search Agent.

    Research the user's question using Google Search.

    Responsibilities:
    1. Find relevant and reliable information.
    2. Answer the user's original question.
    3. Provide a clear initial response.
    4. Include sources when available.
    5. Do not invent facts or citations.

    Your output will be reviewed by the Critique Agent.

    Produce a complete initial answer.
    """,
    tools=[google_search],
    output_key="initial_answer"
)

print("Search Agent created!")


Search Agent created!


**Critique Agent**

In [10]:

critique_agent = Agent(
    name="critique_agent",
    model=MODEL,
    description="Reviews answers and suggests improvements.",
    instruction="""
    You are the Critique Agent.

    Review the following initial answer:

    {initial_answer}

    Evaluate the answer for:

    1. Accuracy
    2. Completeness
    3. Clarity
    4. Relevance to the user's question
    5. Source quality

    Identify any unsupported claims.

    Provide specific recommendations for improvement.

    Do not rewrite the entire answer.

    If the answer is already strong, explain that
    only minor improvements are necessary.
    """,
    output_key="critique_feedback"
)

print("Critique Agent created!")


Critique Agent created!


**Refine Agent**

In [11]:

refine_agent = Agent(
    name="refine_agent",
    model=MODEL,
    description="Improves answers using critique feedback.",
    instruction="""
    You are the Refine Agent.

    Original answer:
    {initial_answer}

    Critique feedback:
    {critique_feedback}

    Your responsibilities:

    1. Review the original answer.
    2. Apply the critique recommendations.
    3. Correct errors when supported by evidence.
    4. Improve clarity and organization.
    5. Preserve accurate facts and valid citations.
    6. Do not invent sources or additional facts.

    Produce the final improved answer.

    Do not include internal critique notes.
    """,
    output_key="final_answer"
)

print("Refine Agent created!")


Refine Agent created!


**Sequential Answer Team**

In [12]:

from google.adk.agents import SequentialAgent

answer_team = SequentialAgent(
    name="answer_team",
    description=(
        "Researches a question, critiques the initial "
        "answer, and produces an improved final response."
    ),
    sub_agents=[
        search_agent,
        critique_agent,
        refine_agent
    ]
)

print("Sequential Answer Team created!")
print("Execution order:")

for agent in answer_team.sub_agents:
    print("-", agent.name)


Sequential Answer Team created!
Execution order:
- search_agent
- critique_agent
- refine_agent


/tmp/ipykernel_119154/2804407436.py:3: DeprecationWarning: SequentialAgent is deprecated in favor of Workflow and will be removed in a future version. Workflow cannot yet be used as an LlmAgent sub-agent.
  answer_team = SequentialAgent(


**Root Agent**

In [13]:

root_agent = Agent(
    name="root_agent",
    model=MODEL,
    description="Coordinates greetings and research workflows.",
    instruction="""
    You are the Root Coordinating Agent.

    You manage two specialized agents:

    1. greeter_agent:
       Handles greetings and introductions.

    2. answer_team:
       Handles questions requiring research,
       evaluation, and refinement.

    Delegation rules:

    - For greetings, delegate to greeter_agent.

    - For factual questions, research requests,
      or information requests, delegate to answer_team.

    - Do not perform the work of specialized agents.

    - Allow the answer_team to complete its full
      sequential workflow.

    - Return the refined answer to the user.
    """,
    sub_agents=[
        greeter_agent,
        answer_team
    ]
)

print("Root Agent:", root_agent.name)
print(
    "Sub-agents:",
    [agent.name for agent in root_agent.sub_agents]
)


Root Agent: root_agent
Sub-agents: ['greeter_agent', 'answer_team']


In [14]:
import uuid
from google.adk.apps.app import App
from google.adk.agents.context_cache_config import ContextCacheConfig
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

# Configure App and Runner parameters with Context Caching enabled
APP_NAME = "seq_query_app"
USER_ID = "student"
SESSION_ID = str(uuid.uuid4())

# Create the App configuration with context caching enabled and pass the required root_agent
app = App(
    name=APP_NAME,
    root_agent=root_agent,
    context_cache_config=ContextCacheConfig()
)

# Initialize the session service (no apps argument needed in this version)
session_service = InMemorySessionService()
await session_service.create_session(app_name=APP_NAME, user_id=USER_ID, session_id=SESSION_ID)
runner = Runner(agent=root_agent, app_name=APP_NAME, session_service=session_service)

# async def ask_agent(prompt):
#     print(f"\nUSER: {prompt}")
#     async for event in runner.run_async(
#         user_id=USER_ID,
#         session_id=SESSION_ID,
#         new_message=types.Content(role="user", parts=[types.Part(text=prompt)]),
#     ):
#         if event.is_final_response() and event.content:
#             for part in event.content.parts or []:
#                 if part.text:
#                     print("AGENT:", part.text)


# # Test routing to the Search Agent
# await ask_agent("What is the capital of Utah?")

# # Test routing to the Weather Agent
# await ask_agent("What is the temperature in Denver?")


**Test Definition**

In [15]:

async def test_agent(prompt):

    print("\n" + "=" * 60)
    print("USER QUESTION:", prompt)
    print("=" * 60)

    async for event in runner.run_async(
        user_id=USER_ID,
        session_id=SESSION_ID,
        new_message=types.Content(
            role="user",
            parts=[types.Part(text=prompt)]
        )
    ):

        print("\nEVENT AUTHOR:", event.author)

        if event.content:
            for part in event.content.parts or []:

                if part.function_call:
                    print(
                        "TOOL CALL:",
                        part.function_call.name
                    )

                if part.function_response:
                    print(
                        "TOOL RESPONSE:",
                        part.function_response.name
                    )

                if part.text:
                    print("OUTPUT:", part.text)

        if event.is_final_response():
            print("FINAL RESPONSE EVENT")

    session = await session_service.get_session(
        app_name=APP_NAME,
        user_id=USER_ID,
        session_id=SESSION_ID
    )

    print("\n--- SHARED SESSION STATE ---")

    for key in [
        "initial_answer",
        "critique_feedback",
        "final_answer"
    ]:
        print(f"\n{key}:")
        print(session.state.get(key, "Not generated"))

    print("\nTEST COMPLETE")


**Test Prompts**

In [16]:

# Test 1: Greeter Agent
await test_agent("Hello!")

# Test 2: Full Sequential Answer Team
await test_agent("What is the capital of Utah?")

# Test 3: Research, Critique, and Refine
await test_agent("What are three benefits of renewable energy?")



USER QUESTION: Hello!

EVENT AUTHOR: root_agent
TOOL CALL: transfer_to_agent

EVENT AUTHOR: root_agent
TOOL RESPONSE: transfer_to_agent

EVENT AUTHOR: greeter_agent
OUTPUT: Hello! Welcome to our AI assistant system. I'm here to help by answering your questions, researching information, reviewing answers for accuracy, and improving responses to ensure you get the best possible assistance.
FINAL RESPONSE EVENT

--- SHARED SESSION STATE ---

initial_answer:
Not generated

critique_feedback:
Not generated

final_answer:
Not generated

TEST COMPLETE

USER QUESTION: What is the capital of Utah?

EVENT AUTHOR: greeter_agent
TOOL CALL: transfer_to_agent

EVENT AUTHOR: greeter_agent
TOOL RESPONSE: transfer_to_agent

EVENT AUTHOR: search_agent
OUTPUT: Salt Lake City is the capital and most populous city of the U.S. state of Utah. It has been the capital of Utah since 1856.
FINAL RESPONSE EVENT

EVENT AUTHOR: critique_agent
OUTPUT: This is a strong and accurate answer.

**1. Accuracy:**
*   The 